# TFG — Modelo 2: Regresión Múltiple

**Objetivo:** estimar la ocupación hotelera (proxy de presión/demanda) a partir de variables  
de calendario, operativa, nieve y estructura. Prioridad: **interpretabilidad** sobre predicción.

**Universo:** 15 estaciones con dato de ocupación hotelera · granularidad mensual · 2018-2025  
**Variable objetivo (Y):** `ocupacion_general_pct` — % plazas hoteleras ocupadas  
**n = 1.097 observaciones** (estación × mes × año)

> **Nota metodológica:** la ocupación hotelera es un *proxy* de demanda, no una medida directa  
> de afluencia a la estación. Sus limitaciones se reconocen explícitamente:  
> (1) granularidad mensual, no diaria; (2) cubre solo municipios con oferta hotelera registrada;  
> (3) mezcla esquiadores con visitantes de otras motivaciones (verano, senderismo, etc.).  
> A pesar de ello, es el mejor proxy disponible en este dataset y es metodológicamente defendible.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import scipy.stats as stats
import warnings, os
warnings.filterwarnings('ignore')

os.makedirs('/content/graficas', exist_ok=True)

C1, C2, C3, C4 = '#2C6E8A', '#E07B39', '#5BA55B', '#C44E52'
PALETA = [C1, C2, C3, C4, '#8172B2', '#937860']

# ── Carga y join ────────────────────────────────────────────────────────────
mm = pd.read_csv('/content/master_mensual.csv')
gt = pd.read_csv('/content/gtrends_clean.csv')
mm = mm.merge(gt[['anio','mes','gt_esquiar']], on=['anio','mes'], how='left')

# ── Subconjunto base: solo filas con ocupación hotelera disponible ───────────
mm_oc = mm[mm['ocupacion_general_pct'].notna()].copy()
mm_oc['post_covid'] = (mm_oc['anio'] >= 2022).astype(int)

print(f"Dataset modelo: {mm_oc.shape[0]} observaciones · {mm_oc['estacion'].nunique()} estaciones")
print(f"Periodo: {mm_oc['anio'].min()}–{mm_oc['anio'].max()}")
print(f"Y (ocupacion_general_pct): media={mm_oc['ocupacion_general_pct'].mean():.1f}%  "
      f"std={mm_oc['ocupacion_general_pct'].std():.1f}  "
      f"rango=[{mm_oc['ocupacion_general_pct'].min():.1f}, {mm_oc['ocupacion_general_pct'].max():.1f}]")

---
## 1. Variable objetivo y justificación

**Y = `ocupacion_general_pct`** — porcentaje medio mensual de plazas hoteleras ocupadas  
en el municipio más cercano a la estación (fuente: INE / Encuesta de Ocupación Hotelera).

**Por qué esta variable:**
- Es el mejor proxy de demanda disponible con cobertura 2018-2025
- Tiene correlación lógica con la presión física en la estación (a mayor ocupación hotelera → más esquiadores potenciales)
- Es continua, sin nulos en el subconjunto de trabajo, y con distribución razonablemente normal
- Alternativas descartadas: `pct_km_abiertos` mide oferta, no demanda; Google Trends tiene r=-0.06 con Y en muestra completa

**Limitación declarada:** la ocupación hotelera incluye visitantes de verano y otras motivaciones,  
lo que genera ruido en meses no-esquí. El modelo lo controla mediante dummies de mes.

In [ ]:
# M1 — Distribución de la variable objetivo ────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Histograma
axes[0].hist(mm_oc['ocupacion_general_pct'], bins=30, color=C1, alpha=0.85, edgecolor='white')
axes[0].axvline(mm_oc['ocupacion_general_pct'].mean(), color=C2, linewidth=2,
                label=f"Media: {mm_oc['ocupacion_general_pct'].mean():.1f}%")
axes[0].set_xlabel('Ocupación general (%)')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Distribución de Y')
axes[0].legend()

# Boxplot por mes
meses_lbl = {1:'Ene',2:'Feb',3:'Mar',4:'Abr',5:'May',6:'Jun',
             7:'Jul',8:'Ago',9:'Sep',10:'Oct',11:'Nov',12:'Dic'}
data_mes = [mm_oc[mm_oc['mes']==m]['ocupacion_general_pct'].values for m in range(1,13)]
bp = axes[1].boxplot(data_mes, patch_artist=True, medianprops=dict(color='white',linewidth=2))
for patch in bp['boxes']:
    patch.set_facecolor(C1); patch.set_alpha(0.7)
axes[1].set_xticklabels([meses_lbl[m] for m in range(1,13)], fontsize=8)
axes[1].set_ylabel('Ocupación general (%)')
axes[1].set_title('Y por mes')

# Boxplot por zona
zonas = sorted(mm_oc['zona_hotelera'].unique())
data_zona = [mm_oc[mm_oc['zona_hotelera']==z]['ocupacion_general_pct'].values for z in zonas]
bp2 = axes[2].boxplot(data_zona, patch_artist=True, medianprops=dict(color='white',linewidth=2))
for patch in bp2['boxes']:
    patch.set_facecolor(C3); patch.set_alpha(0.7)
zonas_short = [z.split()[0] for z in zonas]
axes[2].set_xticklabels(zonas_short, rotation=30, ha='right', fontsize=8)
axes[2].set_ylabel('Ocupación general (%)')
axes[2].set_title('Y por zona hotelera')

plt.suptitle('Modelo 2 — Variable objetivo: ocupación hotelera general', fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('/content/graficas/M2_01_distribucion_Y.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 2. Selección de variables explicativas

### Variables incluidas y justificación

| Variable | Tipo | Justificación |
|---|---|---|
| `pct_dias_abierta` | Operativa | % días del mes que la estación estuvo abierta — mide disponibilidad de oferta |
| `nieve_total_cm` | Meteorología (nieve) | Cm acumulados de nieve en el mes — disponible sin nulos en todo el dataset |
| `dias_semana_navidad` | Calendario festivo | Días de semana navideña en el mes — periodo de alta demanda |
| `dias_semana_santa` | Calendario festivo | Días de Semana Santa en el mes — segundo pico de esquí |
| `dias_finde` | Calendario | Nº de fines de semana en el mes — efecto movilidad corta |
| `post_covid` | Control estructural | Dummy = 1 si año ≥ 2022 — controla el salto de nivel post-pandemia |
| `mes` (dummies) | Estacionalidad | 11 dummies para capturar el patrón anual |
| `zona_hotelera` (dummies) | Efecto localización | 6 dummies — controla diferencias estructurales entre zonas |

### Variables descartadas y por qué

- **`gt_esquiar`**: r = -0.06 con Y en muestra completa; correlación espuria por mezcla temporal  
- **`temp_media_med`**: 66% nulos — reduciría n a 371 observaciones  
- **`pct_km_med`**: 55% nulos — y mide oferta, no demanda  
- **`dias_festivo`**: alta multicolinealidad con `dias_semana_navidad` (r = 0.64) — subsumed  
- **`precipitacion_total`**: r = 0.75 con `nieve_total_cm` — redundante  
- **`altitud`, `orientacion`**: absorbidas por dummies de zona hotelera

In [ ]:
# M2 — Correlaciones entre variables seleccionadas ─────────────────────────
vars_sel = ['dias_finde','dias_semana_navidad','dias_semana_santa',
            'pct_dias_abierta','nieve_total_cm','post_covid','ocupacion_general_pct']
corr_m = mm_oc[vars_sel].corr()

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr_m, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
plt.colorbar(im, ax=ax)
ax.set_xticks(range(len(vars_sel)))
ax.set_yticks(range(len(vars_sel)))
ax.set_xticklabels(vars_sel, rotation=40, ha='right', fontsize=9)
ax.set_yticklabels(vars_sel, fontsize=9)
for i in range(len(vars_sel)):
    for j in range(len(vars_sel)):
        ax.text(j, i, f'{corr_m.iloc[i,j]:.2f}', ha='center', va='center',
                fontsize=8, color='white' if abs(corr_m.iloc[i,j]) > 0.5 else 'black')
ax.set_title('Modelo 2 — Correlaciones entre variables seleccionadas', fontweight='bold')
plt.tight_layout()
plt.savefig('/content/graficas/M2_02_correlaciones_predictores.png', dpi=150, bbox_inches='tight')
plt.show()

print("Correlación con Y (ocupacion_general_pct):")
print(corr_m['ocupacion_general_pct'].drop('ocupacion_general_pct').sort_values(ascending=False).round(3).to_string())

---
## 3. Construcción del modelo

In [ ]:
# M3 — Preparación de datos y construcción del modelo ──────────────────────

# ── Encoding: dummies mes y zona ────────────────────────────────────────────
mes_dummies  = pd.get_dummies(mm_oc['mes'],          prefix='mes',  drop_first=True)
zona_dummies = pd.get_dummies(mm_oc['zona_hotelera'], prefix='zona', drop_first=True)

continuas = ['dias_finde','dias_semana_navidad','dias_semana_santa',
             'pct_dias_abierta','nieve_total_cm','post_covid']

X = pd.concat([mm_oc[continuas].reset_index(drop=True),
               mes_dummies.reset_index(drop=True),
               zona_dummies.reset_index(drop=True)], axis=1).astype(float)
y = mm_oc['ocupacion_general_pct'].reset_index(drop=True)

print(f"Matriz X: {X.shape}  —  {X.shape[1]} variables ({len(continuas)} continuas + "
      f"{mes_dummies.shape[1]} dummies mes + {zona_dummies.shape[1]} dummies zona)")
print(f"Nulos en X: {X.isna().sum().sum()} | Nulos en y: {y.isna().sum()}")

# ── Entrenamiento (modelo completo) ─────────────────────────────────────────
model = LinearRegression()
model.fit(X, y)
y_pred = pd.Series(model.predict(X), index=y.index)

# ── Validación cruzada 5-fold ────────────────────────────────────────────────
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_r2   = cross_val_score(model, X, y, cv=kf, scoring='r2')
cv_mae  = -cross_val_score(model, X, y, cv=kf, scoring='neg_mean_absolute_error')
cv_rmse = np.sqrt(-cross_val_score(model, X, y, cv=kf, scoring='neg_mean_squared_error'))

print(f"\n{'═'*50}")
print(f"  MÉTRICAS DEL MODELO")
print(f"{'═'*50}")
print(f"  R²  (train)   : {r2_score(y, y_pred):.4f}")
print(f"  R²  (CV 5-fold): {cv_r2.mean():.4f} ± {cv_r2.std():.4f}")
print(f"  MAE (CV)      : {cv_mae.mean():.2f} pp ± {cv_mae.std():.2f}")
print(f"  RMSE (CV)     : {cv_rmse.mean():.2f} pp ± {cv_rmse.std():.2f}")
print(f"  n             : {len(y)}")
print(f"{'═'*50}")
print(f"\n  Media Y: {y.mean():.1f}% → MAE relativo: {cv_mae.mean()/y.mean()*100:.1f}% de la media")

---
## 4. Evaluación del modelo

### Lectura de métricas
- **R² CV = 0.40**: el modelo explica el 40% de la varianza de la ocupación hotelera  
  con variables de calendario, nieve y operativa. Es un resultado **modesto pero coherente**  
  dado que el proxy (ocupación hotelera) contiene ruido por motivaciones no-esquí.
- **MAE CV ≈ 8.6 pp**: el error medio es 8-9 puntos porcentuales sobre una media de 45%.  
  Es decir, el modelo se equivoca en promedio ~19% de la media. Aceptable para un proxy.
- **RMSE CV ≈ 10.6 pp**: los errores grandes existen (outliers COVID), pero no son sistemáticos.
- **Estabilidad**: los 5 folds dan R² entre 0.33 y 0.48 → el modelo no es inestable.

In [ ]:
# M4 — Diagnóstico de residuos ──────────────────────────────────────────────
residuos = y - y_pred

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Predicho vs Real
axes[0].scatter(y, y_pred, alpha=0.3, s=15, color=C1)
lim = [min(y.min(), y_pred.min())-2, max(y.max(), y_pred.max())+2]
axes[0].plot(lim, lim, color=C3, linewidth=1.5, linestyle='--', label='Predicción perfecta')
axes[0].set_xlabel('Valor real (%)')
axes[0].set_ylabel('Valor predicho (%)')
axes[0].set_title('Predicho vs Real')
axes[0].legend(fontsize=9)

# Residuos vs Predichos
axes[1].scatter(y_pred, residuos, alpha=0.3, s=15, color=C2)
axes[1].axhline(0, color=C3, linewidth=1.5, linestyle='--')
# Marcar outliers COVID
mask_covid = (mm_oc.reset_index(drop=True)['anio'].isin([2020,2021])) & (residuos.abs() > 20)
axes[1].scatter(y_pred[mask_covid], residuos[mask_covid], color=C4, s=40, zorder=5,
                label='Outliers COVID')
axes[1].set_xlabel('Valor predicho (%)')
axes[1].set_ylabel('Residuo (pp)')
axes[1].set_title('Residuos vs Predichos')
axes[1].legend(fontsize=9)

# Distribución de residuos
axes[2].hist(residuos, bins=35, color=C4, alpha=0.8, edgecolor='white')
axes[2].axvline(0, color=C3, linewidth=1.5, linestyle='--')
axes[2].set_xlabel('Residuo (pp)')
axes[2].set_ylabel('Frecuencia')
axes[2].set_title(f'Distribución de residuos\nmedia={residuos.mean():.2f}  std={residuos.std():.2f}')

plt.suptitle('Modelo 2 — Diagnóstico de residuos', fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('/content/graficas/M2_04_diagnostico_residuos.png', dpi=150, bbox_inches='tight')
plt.show()

# Test normalidad
stat_jb, p_jb = stats.jarque_bera(residuos)
corr_hetero = np.corrcoef(y_pred, np.abs(residuos))[0,1]
print(f"Jarque-Bera: stat={stat_jb:.2f}, p={p_jb:.4f} {'→ ligera no-normalidad' if p_jb<0.05 else '→ OK'}")
print(f"Corr(ŷ, |e|) homocedasticidad: {corr_hetero:.3f} {'→ aceptable' if abs(corr_hetero)<0.3 else '→ revisar'}")
print(f"Outliers (|residuo| > 25pp): {(residuos.abs()>25).sum()} casos (mayoría en 2020-2021 por COVID)")

---
## 5. Interpretación de coeficientes

### Cómo leer los coeficientes
- **Coeficientes RAW**: en unidades originales. Ej: `pct_dias_abierta = +18.2` significa  
  que si la estación pasa de cerrada (0%) a abierta todo el mes (100%), la ocupación  
  hotelera predicha sube +18.2 pp, *manteniendo el resto constante*.
- **Coeficientes beta (estandarizados)**: permiten comparar la **importancia relativa**  
  de cada variable, independientemente de su escala.

In [ ]:
# M5 — Coeficientes estandarizados (betas) ──────────────────────────────────
scaler = StandardScaler()
X_std  = scaler.fit_transform(X)
model_std = LinearRegression().fit(X_std, y)
beta = pd.Series(model_std.coef_, index=X.columns)

# Calcular p-values manualmente (OLS)
n, k = X.shape[0], X.shape[1] + 1
X_aug = np.column_stack([np.ones(n), X.values])
XtX_inv = np.linalg.pinv(X_aug.T @ X_aug)
sigma2  = ((y - y_pred)**2).sum() / (n - k)
se      = np.sqrt(np.diag(XtX_inv) * sigma2)
coefs_all  = np.concatenate([[model.intercept_], model.coef_])
t_stats    = coefs_all / se
p_vals     = 2 * (1 - stats.t.cdf(np.abs(t_stats), df=n-k))
names_all  = ['intercept'] + list(X.columns)
pval_s     = pd.Series(p_vals, index=names_all)

# Tabla resumen: variables continuas + post_covid
tabla_interp = pd.DataFrame({
    'Coef_raw'  : model.coef_,
    'Beta_std'  : beta,
    'p_valor'   : pval_s.drop('intercept'),
}, index=X.columns)
tabla_interp['Significativo'] = tabla_interp['p_valor'].apply(
    lambda p: '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else '')

print("=== VARIABLES CONTINUAS + POST_COVID ===")
vars_clave = ['pct_dias_abierta','nieve_total_cm','post_covid',
              'dias_semana_navidad','dias_semana_santa','dias_finde']
print(tabla_interp.loc[vars_clave][['Coef_raw','Beta_std','p_valor','Significativo']].round(4).to_string())

print("\n=== DUMMIES DE MES (ref: enero) ===")
mes_rows = [c for c in X.columns if c.startswith('mes_')]
print(tabla_interp.loc[mes_rows][['Coef_raw','Beta_std','p_valor','Significativo']].round(4).to_string())

print("\n=== DUMMIES DE ZONA (ref: Benasque) ===")
zona_rows = [c for c in X.columns if c.startswith('zona_')]
print(tabla_interp.loc[zona_rows][['Coef_raw','Beta_std','p_valor','Significativo']].round(4).to_string())

In [ ]:
# M6 — Gráfico de coeficientes beta (variables principales) ─────────────────
# Mostrar solo las 12 variables con mayor |beta| para claridad visual

beta_sorted = beta.abs().sort_values(ascending=False).head(14)
beta_signed = beta[beta_sorted.index]
colores = [C2 if v > 0 else C4 for v in beta_signed]
sig_mark = ['*' if pval_s.get(v, 1) < 0.05 else '' for v in beta_signed.index]
etiquetas = [f"{v}{s}" for v, s in zip(beta_signed.index, sig_mark)]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(etiquetas[::-1], beta_signed.values[::-1], color=colores[::-1], alpha=0.85, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Coeficiente beta estandarizado', fontsize=11)
ax.set_title('Modelo 2 — Importancia relativa de variables\n(azul=positivo, rojo=negativo · *p<0.05)',
             fontweight='bold')
ax.grid(axis='x', alpha=0.3)

# Anotar valores
for bar, val in zip(bars[::-1], beta_signed.values):
    ax.text(val + (0.05 if val >= 0 else -0.05), bar.get_y() + bar.get_height()/2,
            f'{val:.2f}', va='center', ha='left' if val >= 0 else 'right', fontsize=8)

plt.tight_layout()
plt.savefig('/content/graficas/M2_06_coeficientes_beta.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Conclusiones del Modelo 2

### Qué explica la ocupación hotelera (proxy de presión)

**Variables que AUMENTAN la presión (coeficiente positivo, significativo):**

| Variable | Efecto raw | Interpretación |
|---|---|---|
| `mes_8` (agosto) | +29.7 pp | Agosto es el mes de mayor presión, +30pp sobre enero |
| `pct_dias_abierta` | +18.2 pp (por unidad) | Cada 10pp más de días abiertos → +1.8pp de ocupación |
| `mes_7` (julio) | +19.0 pp | Segundo pico estival |
| `post_covid` | +7.7 pp | Desde 2022, la ocupación base es ~8pp mayor que antes |
| `nieve_total_cm` | +0.07 pp por cm | Efecto pequeño pero significativo: la nieve atrae visitantes |

**Variables que REDUCEN la presión (coeficiente negativo, significativo):**

| Variable | Efecto raw | Interpretación |
|---|---|---|
| `zona_Vielha e Mijaran` | -6.7 pp | Baqueira Beret tiene ocupación ~7pp menor que Benasque (referencia) |
| `dias_semana_navidad` | -5.1 pp | Resultado contraintuitivo: ver nota abajo |
| `zona_Sallent de Gállego` | -4.0 pp | Formigal/Panticosa tienen menor ocupación que Benasque/Cerler |

> ⚠️ **Nota sobre `dias_semana_navidad` (no significativo, p=0.10):**  
> El coeficiente negativo parece contradictorio, pero se explica por la presencia de la  
> dummy `mes_12` y `mes_1` que ya capturan el nivel de diciembre/enero.  
> Dentro de esos meses, más días de Navidad no añaden ocupación adicional neta.  
> Este resultado sugiere que el efecto Navidad ya está absorbido por la estacionalidad mensual.

**Variables NO significativas en el modelo:**
- `dias_finde` (p=0.11): el efecto finde es real (ver EDA) pero a nivel mensual  
  queda absorbido por los efectos de mes y operativa
- `dias_semana_santa` (p=0.05): efecto marginal, depende del mes en que cae

### Limitaciones del modelo

1. **R² = 0.40**: el 60% de la varianza queda sin explicar. El modelo es descriptivo,  
   no predictivo de alta precisión. Sus coeficientes son útiles para interpretar,  
   no para hacer forecasting operativo.
2. **COVID distorsiona**: los 15 casos con residuo > 25pp son casi todos de 2020-2021.  
   Considerar excluir o ponderarlos en una versión refinada.
3. **Granularidad mensual**: no captura picos intraday ni variabilidad dentro del mes.

### Implicaciones para el Modelo 3 (sistema de recomendación)

Las reglas del sistema de recomendación deben priorizar:
1. **Mes del año** → es el predictor más potente (agosto +30pp, febrero +1pp sobre enero)
2. **Apertura de la estación** → días operativos predicen demanda
3. **Nieve acumulada** → refuerza la demanda en temporada de invierno
4. **Zona** → Vielha e Mijaran (Baqueira) y Sallent (Formigal) tienen menor presión estructural  
   que Benasque/Cerler → son zonas menos saturadas para el mismo nivel de condiciones

---
## Compresión y descarga

In [ ]:
import zipfile
zip_path = '/content/graficas_modelo2.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir('/content/graficas'):
        if fname.startswith('M2'):
            zf.write(f'/content/graficas/{fname}', fname)
print(f"ZIP generado: {zip_path}")
from google.colab import files
files.download(zip_path)